# Text Generation with Qwen2-1.5B-Instruct

This notebook demonstrates how to use the Qwen2-1.5B-Instruct model for text generation tasks, specifically for generating marketing content.

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

d:\UNI\Semester 7\FYP\TextGeneration\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Model and Tokenizer

Load the Qwen2-1.5B-Instruct model and tokenizer. The model is configured to use GPU acceleration with `torch.float16` precision for optimal performance.

In [2]:
model_name = "Qwen/Qwen2-1.5B-Instruct"  # or local path if downloaded

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"   # this will place weights on GPU where possible
)

d:\UNI\Semester 7\FYP\TextGeneration\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\haide\.cache\huggingface\hub\models--Qwen--Qwen2-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Xet Storage is enabled for this repo, but the 'hf_xe

# System Prompt

In [ ]:
system_prompt = """
You are a professional marketing assistant specializing in fashion, particularly Eastern and traditional clothing.
Your role is to generate creative, persuasive, and culturally relevant marketing content. You can:

1. Write ad copy, slogans, hooks, and call-to-actions for social media, email campaigns, landing pages, and advertisements.
2. Generate social media posts, captions, and short-form content, adapted to the platform (Facebook, Instagram, TikTok, LinkedIn, etc.).
3. Draft blog posts, articles, newsletters, and long-form content for content marketing or SEO purposes.
4. Create marketing campaign ideas, including seasonal, festive, or event-based campaigns, with appropriate themes and messaging.
5. Propose multi-channel strategies combining social media, email, blog, and paid ads.
6. Help maintain brand voice and tone: consistent, culturally relevant, elegant, and modern.
7. Suggest variations and A/B test ideas for ad copy, social media posts, and email sequences.
8. Provide insights on audience targeting, engagement hooks, and creative messaging.
9. Be aware of trends, cultural nuances, and festival seasons when generating marketing ideas.
10. Always produce content that is engaging, persuasive, and suitable for the target audience.

When responding, you should follow these instructions:
- Be creative, professional, and persuasive.
- Respect cultural context and traditions of Eastern clothing.
- Provide multiple variations when possible.
"""


# User Input

In [11]:
# Cell 3 — Prepare a “chat-style” prompt (system + user)
user_input = """
I have a new winter collection of Eastern-inspired jackets and shawls for men and women.
Write a **catchy and persuasive advertisement** for a billboard campaign.
Make it **unique, visually appealing, and culturally relevant**.
Include a **short headline, a creative tagline**, and a **call-to-action** that will attract urban young adults.
Keep it **elegant, stylish, and in line with festive/winter season vibes**.
"""


In [12]:

# Use the Qwen2 chat template for formatting
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_input}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [13]:
# Cell 4 — Tokenize input & generate output
inputs = tokenizer([text], return_tensors="pt").to(model.device)
generated = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)


In [14]:
# Cell 5 — Decode and print output
# We slice off the input-prompt tokens to get only new generation
generated_ids = [output_ids[len(inputs["input_ids"][i]):] 
                 for i, output_ids in enumerate(generated)]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


print("=== Model response ===")
print(response)

=== Model response ===
**Headline:**
"Step into the Eternity of Eastern Style"

**Creative Tagline:**
"Unleash Your Inner Legend with Our Winter Collection!"

**Call-To-Action:**
"Experience the Fusion of Tradition and Modernity - Order Now!"

---

[Image: An elegant man wearing an Eastern-inspired jacket over a shawl, standing amidst snowflakes]

---

[Text:]
In a world where fashion meets heritage, our Winter Collection brings you the epitome of sophistication. Embrace the essence of Eastern style by stepping into your inner legend with our unique blend of tradition and modernity.

Our collection is crafted to captivate the urban youth who appreciate elegance and style. From the moment you slip on one of our jackets or wraps, you'll feel like royalty, blending seamlessly into the cold winter landscape around you.

Discover the fusion of beauty and warmth that our collection offers. It's not just about keeping warm; it's about expressing yourself through timeless craftsmanship and cul